<img src="../../../figs/holberton_logo.png" alt="logo" width="500"/>

# Natural Language Processing - Evaluation Metrics

## 0. Unigram BLEU score

Write the function `def uni_bleu(references, sentence)`: that calculates the unigram BLEU score for a sentence:

- `references` is a list of reference translations


- each reference translation is a list of the words in the translation


- `sentence` is a list containing the model proposed sentence


- Returns: the unigram BLEU score

In [1]:
# FOR ILLUSTRATION PURPOSES AS THE NLTK MODULE IS NOT ALLOWED TO USE

import nltk
from nltk.translate.bleu_score import sentence_bleu

def uni_bleu(references, sentence):
    # Calculate BLEU score with unigram (1-gram) precision
    bleu_score = sentence_bleu(references, sentence, weights=(1, 0, 0, 0))
    return bleu_score

In [2]:
references = [["the", "cat", "is", "on", "the", "mat"], ["there", "is", "a", "cat", "on", "the", "mat"]]
sentence = ["there", "is", "a", "cat", "here"]
print(uni_bleu(references, sentence))

0.6549846024623855


### Description of BLEU Score

BLEU (Bilingual Evaluation Understudy) is an evaluation metric used to **assess the quality of machine-generated text by comparing it to one or more reference translations**. It is commonly used in tasks like machine translation. 

BLEU score **measures the precision of n-grams** (e.g., unigrams, bigrams) in the generated text compared to the reference texts.

### Formula for BLEU Score:

#### 1. Brevity Penalty (BP)

To penalize short translations, BP is used to adjust the score if the generated text is shorter than the reference texts. The formula for BP is:

$$
BP = \begin{cases} 
1 & \text{if } \text{len}(s) > \text{len}(r) \\ 
\exp\left(1 - \frac{\text{len}(r)}{\text{len}(s)}\right) & \text{otherwise} 
\end{cases}
$$

where $len(s)$ is the length of the generated sentence and $len(r)$ is the length of the closest reference sentence.

#### 2. Precision Calculation

- **Purpose**: To calculate the unigram precision, which is the proportion of unigrams in the generated sentence that appear in the reference sentences.
- **Details** 
    - `set(sentence)` ensures each word in the generated sentence is considered once. 
    - `sum(match in reference for match in set(sentence))` counts the number of unigrams in the sentence that appear in each reference. 
    - `max(...)` takes the highest count across all references to handle multiple references
    
#### 3. BLEU Score Calculation:

BLEU score combines precision and brevity penalty

$$
\text{BLEU} = BP \times \exp\left(\sum_{n=1}^{N} w_n \cdot \log(p_n)\right)
$$

where $p(n)$ is the precision for n-grams, and  $w_n$ is the weight for n-grams (usually $1/N$ for each $n$).

In [3]:
import numpy as np


def uni_bleu(references, sentence):
    
    BP = min(1, np.exp(1 - len(min(references, key=len)) / len(sentence)))

    precision = max([sum(match in reference for match in set(sentence))
                     for reference in references]) / len(sentence)

    return BP * np.exp(np.log(precision))

In [4]:
references = [["the", "cat", "is", "on", "the", "mat"], ["there", "is", "a", "cat", "on", "the", "mat"]]
sentence = ["there", "is", "a", "cat", "here"]
print(uni_bleu(references, sentence))

0.6549846024623855


## 1. N-gram BLEU score

Write the function `def ngram_bleu(references, sentence, n)`: that calculates the n-gram BLEU score for a sentence:

- `references` is a list of reference translations


- each reference translation is a list of the words in the translation


- `sentence` is a list containing the model proposed sentence


- `n` is the size of the n-gram to use for evaluation


- Returns: the n-gram BLEU score

In [5]:
import nltk
from nltk.translate.bleu_score import sentence_bleu

def ngram_bleu(references, sentence, n):
    # Calculate BLEU score with n-gram precision
    bleu_score = sentence_bleu(references, sentence, weights=(1/n,) * n)
    return bleu_score

In [6]:
references = [["the", "cat", "is", "on", "the", "mat"], ["there", "is", "a", "cat", "on", "the", "mat"]]
sentence = ["there", "is", "a", "cat", "here"]

print(ngram_bleu(references, sentence, 2))

0.6341861143397762


### N-Gram BLEU Score

The n-gram BLEU score **measures the quality of the generated text by comparing the n-grams in the generated sentence to those in the reference translations**. 

It evaluates **how many n-grams** (sequences of n words) in the generated text **appear in the reference texts**. The BLEU score is adjusted using a brevity penalty to account for shorter sentences.

#### Brevity Penalty (BP)

$$
BP = \begin{cases} 
1 & \text{if } \text{len}(s) > \text{len}(r) \\
\exp\left(1 - \frac{\text{len}(r)}{\text{len}(s)}\right) & \text{otherwise} 
\end{cases}
$$

#### n-gram BLEU Score

$$
\text{BLEU} = BP \times \exp\left(\frac{1}{N} \sum_{n=1}^{N} \log(p_n)\right)
$$

where $p_n$ is the precision for n-grams of size $n$


In [7]:
import numpy as np


def ngram_bleu(references, sentence, n):

    BP = min(1, np.exp(1 - len(min(references, key=len)) / len(sentence)))
    n_grams = []
    n_grams_ref = 0

    for reference in references:
        n_grams_ref = []
        for i in range(len(sentence) - (n - 1)):
            if any(sentence[i:i + n] == reference[j:j+n]
                   for j in range(len(reference) - (n - 1))) and \
                    sentence[i:i+n] not in n_grams_ref:
                n_grams_ref.append(sentence[i:i+n])
        n_grams.append(len(n_grams_ref))

    precision = max(n_grams) / (i + 1)

    return BP * np.exp(np.log(precision))

### Key Steps

#### Calculate Brevity Penalty (BP)

- Ppenalize short sentences compared to the reference sentences.
- The length of the shortest reference sentence (`len(min(references, key=len))`) is compared to the length of the generated sentence (`len(sentence)`). If the generated sentence is shorter, the `BP` is computed using the exponential formula; otherwise, BP is `1`

```python
BP = min(1, np.exp(1 - len(min(references, key=len)) / len(sentence)))
```

#### Initialize Lists for n-grams

To store counts of matching n-grams between the sentence and reference translations.

```python
n_grams = []
n_grams_ref = 0
```

#### Extract n-grams from Reference Translations
- Identify and count `n-grams` in the generated sentence that match `n-grams` in the reference translations
- For each reference, the code extracts `n-grams` of size `n` from both the `sentence` and `reference`. It then checks if these n-grams match and adds unique matches to `n_grams_ref`. The length of `n_grams_ref` is appended to the `n_grams` list

```python
for reference in references:
    n_grams_ref = []
    for i in range(len(sentence) - (n - 1)):
        if any(sentence[i:i + n] == reference[j:j+n]
               for j in range(len(reference) - (n - 1))) and \
            sentence[i:i+n] not in n_grams_ref:
            n_grams_ref.append(sentence[i:i+n])
    n_grams.append(len(n_grams_ref))

```
#### Calculate Precision

- Compute the precision of n-grams in the sentence.
- `max(n_grams)` takes the highest count of matching `n-grams` found across all references. 
- `i + 1` is the total number of n-grams in the generated sentence. 
- Precision is the ratio of matched n-grams to total n-grams

```python
precision = max(n_grams) / (i + 1)
```

#### Calculate the Final BLEU Score

- Combine the brevity penalty and precision into the final BLEU score.
- The precision is logarithmically transformed and then exponentiated to match the BLEU score formula. The result is multiplied by the brevity penalty (BP).

```python
return BP * np.exp(np.log(precision))
```

In [8]:
references = [["the", "cat", "is", "on", "the", "mat"], ["there", "is", "a", "cat", "on", "the", "mat"]]
sentence = ["there", "is", "a", "cat", "here"]

print(ngram_bleu(references, sentence, 2))

0.6140480648084865


## 2. Cumulative N-gram BLEU score

Write the function `def cumulative_bleu(references, sentence, n)`: that calculates the cumulative n-gram BLEU score for a sentence:

- `references` is a list of reference translations


- each reference translation is a list of the words in the translation


- `sentence` is a list containing the model proposed sentence


- `n` is the size of the largest n-gram to use for evaluation


- All `n-gram` scores should be weighted evenly


- Returns: the cumulative n-gram BLEU score

In [9]:
import nltk
from nltk.translate.bleu_score import sentence_bleu

def cumulative_bleu(references, sentence, n):
    weights = [(1/n,) * n]  # Equal weights for all n-grams
    bleu_score = sentence_bleu(references, sentence, weights=weights)
    return bleu_score

def ngram_bleu(references, sentence, n):
    # Calculate BLEU score with n-gram precision
    bleu_score = sentence_bleu(references, sentence, weights=(1/n,) * n)
    return bleu_score

In [10]:
references = [["the", "cat", "is", "on", "the", "mat"], ["there", "is", "a", "cat", "on", "the", "mat"]]
sentence = ["there", "is", "a", "cat", "here"]

print(cumulative_bleu(references, sentence, 4))

0.5475182535069453


### Cumulative N-Gram BLEU Score VERSUS N-Gram BLEU Score

The N-gram BLEU score measures how well a generated sentence matches reference translations for a specific n-gram size, n. It checks how many n-grams of that size appear in both the generated sentence and the references, producing a single score for that n-gram size.

The Cumulative n-gram BLEU score looks at multiple n-gram sizes, from 1 up to n. It calculates precision for each n-gram size and then averages these precision scores. This method gives a broader assessment by evaluating how well the sentence performs across various n-gram sizes.

### Formulas

#### Brevity Penalty

$$
BP = \begin{cases} 
1 & \text{if } \text{len}(s) > \text{len}(r) \\
\exp\left(1 - \frac{\text{len}(r)}{\text{len}(s)}\right) & \text{otherwise} 
\end{cases}
$$

#### Cumulative n-gram BLEU Score

$$
\text{BLEU} = BP \times \exp\left(\frac{1}{N} \sum_{m=1}^{N} \log(p_m)\right)
$$

In [13]:
#!/usr/bin/env python3
"""Cumulative N-gram BLEU Score"""
import numpy as np


def cumulative_bleu(references, sentence, n):

    BP = min(1, np.exp(1 - len(min(references, key=len)) / len(sentence)))
    precision = []

    for m in range(1, n+1):
        n_grams = []
        for reference in references:
            n_grams_ref = []
            for i in range(len(sentence) - (m - 1)):
                if any(sentence[i:i + m] == reference[j:j+m]
                       for j in range(len(reference) - (m - 1))) and \
                        sentence[i:i+m] not in n_grams_ref:
                    n_grams_ref.append(sentence[i:i+m])
            n_grams.append(len(n_grams_ref))
        precision.append(max(n_grams) / (i + 1))

    return BP * np.exp(np.mean(np.log(precision)))

In [14]:
references = [["the", "cat", "is", "on", "the", "mat"], ["there", "is", "a", "cat", "on", "the", "mat"]]
sentence = ["there", "is", "a", "cat", "here"]

print(cumulative_bleu(references, sentence, 4))

0.5475182535069453


### Happy Coding